In [1]:
from ngboost import NGBRegressor
from ngboost.distns import T
from sklearn.tree import DecisionTreeRegressor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/processed")

model_df = pd.read_csv(
    DATA_DIR / "model_data.csv"
)

In [3]:
def prepare_next_month_dataset(model_df: pd.DataFrame):

    df = model_df.copy()

    df = df.sort_values(
        ["set", "card", "t"]
    )

    df["next_price"] = (
        df
        .groupby(["set", "card"])["current_price"]
        .shift(-1)
    )

    valid_prices = (
        (df["current_price"] > 0) &
        (df["next_price"] > 0) &
        np.isfinite(df["current_price"]) &
        np.isfinite(df["next_price"])
    )

    df["next_log_return"] = np.nan

    df.loc[valid_prices, "next_log_return"] = (
        np.log(
            df.loc[valid_prices, "next_price"] /
            df.loc[valid_prices, "current_price"]
        )
    )

    df = df[
        np.isfinite(df["next_log_return"])
    ].copy()

    return df

def add_price_features(df: pd.DataFrame):
    df = df.copy()

    group = df.groupby(["set", "card"])

    # Previous returns
    df["log_return_1m"] = (
        group["current_price"]
        .transform(lambda x: np.log(x / x.shift(1)))
    )

    df["log_return_2m"] = (
        group["current_price"]
        .transform(lambda x: np.log(x / x.shift(2)))
    )

    df["log_return_3m"] = (
        group["current_price"]
        .transform(lambda x: np.log(x / x.shift(3)))
    )

    # Price momentum
    df["momentum_3m"] = (
        group["current_price"]
        .transform(lambda x: x / x.shift(3) - 1)
    )

    # Rolling volatility
    df["volatility_3m"] = (
        group["log_return_1m"]
        .transform(lambda x: x.rolling(3).std())
    )

    # Card age
    df["card_age"] = df["t"]

    return df

In [ ]:
df = prepare_next_month_dataset(model_df)
df = add_price_features(df)

g = df.groupby(["set", "card"])

df["log_current_price"] = np.log1p(df["current_price"])

df["return_since_release"] = np.log(
    df["current_price"] / df["price_on_release"]
)

df["log_return_1m"] = g["current_price"].transform(
    lambda x: np.log(x / x.shift(1))
)

df["log_return_3m"] = g["current_price"].transform(
    lambda x: np.log(x / x.shift(3))
)

df["log_return_6m"] = g["current_price"].transform(
    lambda x: np.log(x / x.shift(6))
)

df["momentum_3m"] = g["current_price"].transform(
    lambda x: x / x.shift(3) - 1
)

df["momentum_6m"] = g["current_price"].transform(
    lambda x: x / x.shift(6) - 1
)

df["price_mean_3m"] = g["current_price"].transform(
    lambda x: x.shift(1).rolling(3).mean()
)

df["price_mean_6m"] = g["current_price"].transform(
    lambda x: x.shift(1).rolling(6).mean()
)

df["price_vs_mean_3m"] = (
    df["current_price"] / df["price_mean_3m"]
)

df["price_vs_mean_6m"] = (
    df["current_price"] / df["price_mean_6m"]
)

df["volatility_3m"] = g["current_price"].transform(
    lambda x: np.log(x / x.shift(1)).rolling(3).std()
)

df["volatility_6m"] = g["current_price"].transform(
    lambda x: np.log(x / x.shift(1)).rolling(6).std()
)

df["history_length"] = g.cumcount()

df["popularity_chase"] = (
    df["popularity_index"] * df["is_chase"]
)

df["ebay_chase"] = (
    df["ebay_index"] * df["is_chase"]
)

df["set_popularity_chase"] = (
    df["set_popularity_index"] * df["is_chase"]
)

In [5]:
df.to_csv("data/model.csv", index=False)